In [1]:
import pandas as pd

# ── Load files ──────────────────────────────────────────────────────────────
# prompts  = pd.read_csv('journal_prompts_with_category.csv')
prompts  = pd.read_csv('journal_prompts_with_category_viewed.csv')
journals = pd.read_csv('journals_with_week_labels.csv')

# ── Normalize dates to YYYY-MM-DD ────────────────────────────────────────────
prompts['date']  = pd.to_datetime(prompts['day']).dt.date.astype(str)
journals['date'] = pd.to_datetime(journals['day']).dt.date.astype(str)

# ── Rename columns for clarity ───────────────────────────────────────────────
prompts  = prompts.rename(columns={'uid': 'participant_id', 'prompt': 'prompt_text'})
journals = journals.rename(columns={
    'uid':          'participant_id',
    'data':         'journal_response_text',
    'week_per_uid': 'week'
})

# ── Deduplicate ──────────────────────────────────────────────────────────────
# Prompts: each (uid, date, prompt_text) appears 3× — collapse to unique rows
prompts_dedup = prompts.drop_duplicates(subset=['participant_id', 'date', 'prompt_text'])

# Journals: one entry per participant per day
journals_dedup = (
    journals[['participant_id', 'date', 'week', 'journal_response_text']]
    .drop_duplicates(subset=['participant_id', 'date'])
)

# ── Merge ─────────────────────────────────────────────────────────────────────
# Left join: keep all prompts; null where no journal entry exists for that day
merged = prompts_dedup.merge(journals_dedup, on=['participant_id', 'date'], how='left')

# ── Final column order & rename ───────────────────────────────────────────────
merged = (
    merged[['participant_id', 'week', 'date', 'prompt_text', 'journal_response_text', 'category']]
    .rename(columns={'category': 'behavioral_domain_category'})
    .sort_values(['participant_id', 'date'])
    .reset_index(drop=True)
)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Total rows       : {len(merged)}")
print(f"Missing responses: {merged['journal_response_text'].isna().sum()}")
print(merged.head())

# ── Export ────────────────────────────────────────────────────────────────────
merged.to_csv('prompts_with_journal_responses_task2.csv', index=False)
print("\nSaved → prompts_with_journal_responses_task2.csv")

Total rows       : 648
Missing responses: 60
   participant_id  week        date  \
0  t0002@sreflect   1.0  2024-02-01   
1  t0002@sreflect   1.0  2024-02-02   
2  t0002@sreflect   1.0  2024-02-04   
3  t0002@sreflect   1.0  2024-02-05   
4  t0002@sreflect   1.0  2024-02-07   

                                         prompt_text  \
0  Your mobile trends show less phone use and dec...   
1  You've embraced more face-to-face chats and le...   
2  Seeing your digital habits improve, envision h...   
3  You've upped your social game—how has this ref...   
4  Your screen interactions have gone up recently...   

                               journal_response_text  \
0  I've been really busy recently so I have not b...   
1  I've been traveling with my partner today so I...   
2  I hope to be more productive next week by spen...   
3  I do not know if I "upped" my social game toda...   
4  I did not really have a positive online intera...   

  behavioral_domain_category  
0             d